# Tracing the AI agent

## How tracing works:

Your LLM application handles requests — each one might call a model, retrieve documents, run tools, and return a response. A trace captures that entire journey as a tree of spans. Each span represents one operation (an LLM call, a retrieval, a tool invocation) with its input, output, timing, and metadata.

In [1]:
from openai import OpenAI
import pandas as pd
import json
import duckdb
from pydantic import BaseModel, Field
from IPython.display import Markdown

from helper import get_openai_api_key, get_phoenix_endpoint

import warnings
warnings.filterwarnings("ignore")

In [2]:
import phoenix as px
import os
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor
from openinference.semconv.trace import SpanAttributes
from opentelemetry.trace import Status, StatusCode
from openinference.instrumentation import TracerProvider

In [3]:
openai_api_key = get_openai_api_key()
client = OpenAI(api_key = openai_api_key)

MODEL = "gpt-4o-mini"

In [4]:
PROJECT_NAME = "tracing-agent" # this is to isolate the specific test runs

In [5]:
tracer_provider = register(
    project_name = PROJECT_NAME,
    endpoint = get_phoenix_endpoint() + "v1/traces"
)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: tracing-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://s172-29-34-111p6006.lab-aws-production.deeplearning.ai/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



1. register(...): This function hooks into your Python environment and installs a "flight recorder" (an OpenTelemetry tracer). Once registered, it silently watches your LLM framework (like LangChain, LangGraph, or the OpenAI SDK) and records every single step the agent takes.
2. endpoint: This is the destination URL where the recorder sends the data. By appending "v1/traces", you are telling the tracer to send the execution logs to the specific port where the Phoenix server is listening for trace data.
3. The string v1/traces is a standardized HTTP endpoint path used by OpenTelemetry to route execution data to the correct processing engine on a server.

In [6]:
OpenAIInstrumentor().instrument(tracer_provider = tracer_provider) #this only collects the openai api calls

In [7]:
tracer = tracer_provider.get_tracer(__name__) #get_tracer is the precise label maker for the telemetry data

## setting span order

tracing telemetry should be built like a tree in hierarchical manner. You must establish the "root span" (the overarching agent execution) before you can track the "child spans" (individual tool executions).
Otherwise, the tool actions would be recorded as floating, disconnected fragments. So we will start with run_agent function which is the root span.

In [8]:
TRANSACTION_DATA_FILE_PATH = 'data/Store_Sales_Price_Elasticity_Promotions_Data.parquet'

In [9]:
SQL_GENERATION_PROMPT = """
Generate an SQL query based on a prompt. Do not reply with anything besides the SQL query.
The prompt is: {prompt}

The available columns are: {columns}
The table name is: {table_name}
"""

In [10]:
def generate_sql_query(prompt: str, columns: list, table_name: str) -> str:
    """Generate an SQL query based on a prompt"""
    formatted_prompt = SQL_GENERATION_PROMPT.format(prompt=prompt, 
                                                    columns=columns, 
                                                    table_name=table_name)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    return response.choices[0].message.content

The Decorator (@tracer.tool): This is a blunt, automatic instrument. It blindly captures the exact arguments passed into the function and the exact final return value. It is perfect for simple tools where the inputs and outputs are just short strings or standard JSON.

The Context Block (with tracer.start_as_current_span...): This is a precision instrument. You use manual tracing when the automatic decorator would capture too much, too little, or the wrong thing entirely.

In [11]:
@tracer.tool()
def lookup_sales_data(prompt: str) -> str:
    """Implementation of sales data lookup from parquet file using SQL"""
    try:

        table_name = "sales"
        
        df = pd.read_parquet(TRANSACTION_DATA_FILE_PATH)
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM df")

        sql_query = generate_sql_query(prompt, df.columns, table_name)
        sql_query = sql_query.strip()
        sql_query = sql_query.replace("```sql", "").replace("```", "")
        
        #this creates span inside another span
        with tracer.start_as_current_span(
            "execute_sql_query", 
            openinference_span_kind="chain"
        ) as span:
            span.set_input(sql_query)
            result = duckdb.sql(sql_query).df()
            span.set_output(value=str(result))
            span.set_status(StatusCode.OK)
        
        return result.to_string()
    except Exception as e:
        return f"Error accessing data: {str(e)}"

In [12]:
DATA_ANALYSIS_PROMPT = """
Analyze the following data: {data}
Your job is to answer the following question: {prompt}
"""

In [13]:
@tracer.tool()
def analyze_sales_data(prompt: str, data: str) -> str:
    """Implementation of AI-powered sales data analysis"""
    formatted_prompt = DATA_ANALYSIS_PROMPT.format(data=data, prompt=prompt)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    analysis = response.choices[0].message.content
    return analysis if analysis else "No analysis could be generated"

In [14]:
CHART_CONFIGURATION_PROMPT = """
Generate a chart configuration based on this data: {data}
The goal is to show: {visualization_goal}
"""

In [15]:
class VisualizationConfig(BaseModel):
    chart_type: str = Field(..., description="Type of chart to generate")
    x_axis: str = Field(..., description="Name of the x-axis column")
    y_axis: str = Field(..., description="Name of the y-axis column")
    title: str = Field(..., description="Title of the chart")

In [16]:
@tracer.chain()
def extract_chart_config(data: str, visualization_goal: str) -> dict:
    """Generate chart visualization configuration
    
    Args:
        data: String containing the data to visualize
        visualization_goal: Description of what the visualization should show
        
    Returns:
        Dictionary containing line chart configuration
    """
    formatted_prompt = CHART_CONFIGURATION_PROMPT.format(data=data, 
                                                         visualization_goal=visualization_goal)
    
    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
        response_format=VisualizationConfig,
    )
    
    try:
        # Extract axis and title info from response
        content = response.choices[0].message.content
        
        # Return structured chart config
        return {
            "chart_type": content.chart_type,
            "x_axis": content.x_axis,
            "y_axis": content.y_axis,
            "title": content.title,
            "data": data
        }
    except Exception:
        return {
            "chart_type": "line", 
            "x_axis": "date",
            "y_axis": "value",
            "title": visualization_goal,
            "data": data
        }

In [17]:
CREATE_CHART_PROMPT = """
Write python code to create a chart based on the following configuration.
Only return the code, no other text.
config: {config}
"""

In [18]:
@tracer.chain()
def create_chart(config: dict) -> str:
    """Create a chart based on the configuration"""
    formatted_prompt = CREATE_CHART_PROMPT.format(config=config)
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    code = response.choices[0].message.content
    code = code.replace("```python", "").replace("```", "")
    code = code.strip()
    
    return code

In [19]:
@tracer.tool()
def generate_visualization(data: str, visualization_goal: str) -> str:
    """Generate a visualization based on the data and goal"""
    config = extract_chart_config(data, visualization_goal)
    code = create_chart(config)
    return code

In [20]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_sales_data",
            "description": "Look up data from Store Sales Price Elasticity Promotions dataset",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_sales_data", 
            "description": "Analyze sales data to extract insights",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool's output."},
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["data", "prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_visualization",
            "description": "Generate Python code to create data visualizations",
            "parameters": {
                "type": "object", 
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool's output."},
                    "visualization_goal": {"type": "string", "description": "The goal of the visualization."}
                },
                "required": ["data", "visualization_goal"]
            }
        }
    }
]

# Dictionary mapping function names to their implementations
tool_implementations = {
    "lookup_sales_data": lookup_sales_data,
    "analyze_sales_data": analyze_sales_data, 
    "generate_visualization": generate_visualization
}

In [21]:
@tracer.chain()
def handle_tool_calls(tool_calls, messages):
    
    for tool_call in tool_calls:   
        function = tool_implementations[tool_call.function.name]
        function_args = json.loads(tool_call.function.arguments)
        result = function(**function_args)
        messages.append({"role": "tool", 
                         "content": result, 
                         "tool_call_id": tool_call.id})
        
    return messages

In [22]:
SYSTEM_PROMPT = """
You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.
"""

In [23]:
def run_agent(messages):
    print("Running agent with messages:", messages)
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not any(
            isinstance(message, dict) and message.get("role") == "system" for message in messages
        ):
            system_prompt = {"role": "system", "content": SYSTEM_PROMPT}
            messages.append(system_prompt)

    while True:
        # Router Span
        print("Starting router call span")
            
        with tracer.start_as_current_span("router_call", openinference_span_kind = "chain") as span:
            response = client.chat.completions.create(
                model = MODEL,
                messages = messages,
                tools = tools,
            )
            
            messages.append(response.choices[0].message.model_dump())
            tool_calls = response.choices[0].message.tool_calls
            print("Received response with tool calls:", bool(tool_calls))
            span.set_status(StatusCode.OK)
    
            if tool_calls:
                print("Starting tool calls span")
                messages = handle_tool_calls(tool_calls, messages)
                span.set_output(value=tool_calls)
            else:
                print("No tool calls, returning final response")
                span.set_output(value=response.choices[0].message.content)
                return response.choices[0].message.content

## Tracing the main agent run:
* start_as_current_span("span-name", ...): this starts the timer automatically when the agent begins and stops when agent run finishes
* The set_status method acts as a diagnostic flag that visually paints the execution block green (success) or red (error) in your Arize Phoenix dashboard.

In [24]:
def start_main_span(messages):
    print(f"starting the main span with messages: {messages}")
    
    with tracer.start_as_current_span("agentrun", openinference_span_kind = "agent") as span:
        span.set_input(value = messages)
        ret = run_agent(messages)
        print("main span completed with return value: ", ret)
        span.set_output(value = ret)
        span.set_status(StatusCode.OK)
        return ret

In [25]:
result = start_main_span([{"role": "user", "content": "Which stores did the best in 2021?"}])

starting the main span with messages: [{'role': 'user', 'content': 'Which stores did the best in 2021?'}]
Running agent with messages: [{'role': 'user', 'content': 'Which stores did the best in 2021?'}]
Starting router call span
Received response with tool calls: True
Starting tool calls span
Starting router call span
Received response with tool calls: False
No tool calls, returning final response
main span completed with return value:  In 2021, the stores that performed the best based on total sales are as follows:

1. **Store 2970**: $84,454.33
2. **Store 3300**: $63,205.33
3. **Store 1650**: $62,152.43
4. **Store 1540**: $58,777.02
5. **Store 1210**: $55,435.62
6. **Store 1100**: $52,494.89
7. **Store 1320**: $52,407.36
8. **Store 2750**: $51,959.20
9. **Store 2200**: $51,634.51
10. **Store 3080**: $50,704.06

These stores led in total sales for the year.
